# DNS Tunneling Detection

Detect synthetic tunneling-like DNS queries using lexical and behavioral features.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Measure query entropy, train a transparent classifier, and rank the most suspicious domains.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def sigmoid(values):
    clipped = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-clipped))

def split_indices(size, test_fraction=0.25):
    shuffled = rng.permutation(size)
    split_at = int(size * (1 - test_fraction))
    return shuffled[:split_at], shuffled[split_at:]

def standardize(train_values, test_values):
    mean = train_values.mean(axis=0)
    std = train_values.std(axis=0)
    std = np.where(std < 1e-9, 1.0, std)
    return (train_values - mean) / std, (test_values - mean) / std, mean, std

def fit_logistic(features, labels, steps=1400, learning_rate=0.08, l2=0.01):
    design = np.column_stack([np.ones(len(features)), features])
    weights = np.zeros(design.shape[1])
    for _ in range(steps):
        probabilities = sigmoid(design @ weights)
        gradient = design.T @ (probabilities - labels) / len(labels)
        gradient[1:] += l2 * weights[1:]
        weights -= learning_rate * gradient
    return weights

def predict_probability(features, weights):
    design = np.column_stack([np.ones(len(features)), features])
    return sigmoid(design @ weights)

def classification_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    tp = int(((labels == 1) & (predictions == 1)).sum())
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    accuracy = (tp + tn) / max(len(labels), 1)
    return pd.Series({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    })


import math
import string
from collections import Counter

def shannon_entropy(value):
    counts = Counter(value)
    probabilities = [count / len(value) for count in counts.values()]
    return -sum(probability * math.log2(probability) for probability in probabilities)


## Steps

### 1. Generate DNS query telemetry


In [2]:
alphabet = np.array(list(string.ascii_lowercase + string.digits))
normal_subdomains = ["www", "api", "mail", "cdn", "login", "docs", "status", "images"]
records = []

for index in range(900):
    is_tunnel = int(rng.random() < 0.16)
    if is_tunnel:
        length = int(rng.integers(32, 70))
        subdomain = "".join(rng.choice(alphabet, size=length))
        query_rate = int(rng.poisson(55) + 15)
        unique_ratio = float(rng.uniform(0.72, 1.0))
        txt_query = int(rng.random() < 0.42)
    else:
        base = str(rng.choice(normal_subdomains))
        suffix = str(rng.integers(1, 80)) if rng.random() < 0.18 else ""
        subdomain = base + suffix
        query_rate = int(rng.poisson(8))
        unique_ratio = float(rng.uniform(0.04, 0.45))
        txt_query = int(rng.random() < 0.03)

    records.append({
        "query": f"{subdomain}.example.test",
        "subdomain_length": len(subdomain),
        "entropy": shannon_entropy(subdomain),
        "digit_ratio": sum(character.isdigit() for character in subdomain) / len(subdomain),
        "query_rate": query_rate,
        "unique_ratio": unique_ratio,
        "txt_query": txt_query,
        "tunnel": is_tunnel,
    })

dns_queries = pd.DataFrame(records)
print("Query count:", len(dns_queries))
print("Tunneling rate:", round(dns_queries["tunnel"].mean(), 3))
print(dns_queries.sample(6, random_state=SEED).round(3).to_string(index=False))


Query count: 900
Tunneling rate: 0.166
              query  subdomain_length  entropy  digit_ratio  query_rate  unique_ratio  txt_query  tunnel
 login.example.test                 5    2.322        0.000          12         0.195          0       0
mail56.example.test                 6    2.585        0.333          13         0.180          0       0
  mail.example.test                 4    2.000        0.000          10         0.200          0       0
status.example.test                 6    1.918        0.000           8         0.079          0       0
   cdn.example.test                 3    1.585        0.000           1         0.394          1       0
mail73.example.test                 6    2.585        0.333          10         0.298          0       0


### 2. Train the detector and rank queries


In [3]:
feature_names = ["subdomain_length", "entropy", "digit_ratio", "query_rate", "unique_ratio", "txt_query"]
train_rows, test_rows = split_indices(len(dns_queries))
train_values = dns_queries.loc[train_rows, feature_names].to_numpy(float)
test_values = dns_queries.loc[test_rows, feature_names].to_numpy(float)
train_labels = dns_queries.loc[train_rows, "tunnel"].to_numpy(int)
test_labels = dns_queries.loc[test_rows, "tunnel"].to_numpy(int)

train_scaled, test_scaled, _, _ = standardize(train_values, test_values)
dns_weights = fit_logistic(train_scaled, train_labels)
dns_probability = predict_probability(test_scaled, dns_weights)
dns_prediction = (dns_probability >= 0.5).astype(int)
dns_metrics = classification_metrics(test_labels, dns_prediction)

ranked_dns = dns_queries.loc[test_rows].copy()
ranked_dns["risk_probability"] = dns_probability
ranked_dns = ranked_dns.sort_values("risk_probability", ascending=False)
dns_importance = pd.DataFrame({
    "feature": feature_names,
    "standardized_weight": dns_weights[1:],
}).sort_values("standardized_weight", ascending=False)

print("Test metrics:")
print(dns_metrics.round(3).to_string())
print("\nFeature weights:")
print(dns_importance.round(3).to_string(index=False))
print("\nHighest-risk queries:")
print(ranked_dns.head(8).round(3).to_string(index=False))


Test metrics:
accuracy       1.0
precision      1.0
recall         1.0
f1             1.0
tp            33.0
fp             0.0
tn           192.0
fn             0.0

Feature weights:
         feature  standardized_weight
      query_rate                1.040
subdomain_length                0.967
    unique_ratio                0.791
         entropy                0.609
       txt_query                0.209
     digit_ratio                0.196

Highest-risk queries:
                                                                             query  subdomain_length  entropy  digit_ratio  query_rate  unique_ratio  txt_query  tunnel  risk_probability
   7qgnmjk0dlqt57ktawfcygrd01u4pgfhhlhvfwmys6pkvxpuqll6yd8zb0v6rsllw3.example.test                66    4.763        0.197          76         0.935          1       1             0.998
   87iefwbdxfcaznkxgsrywyhftcqw3x9uqiy3lcawxeaxopjz0bdtvffyqevcizic8i.example.test                66    4.628        0.106          78         0.959       

## Checks


In [4]:
assert dns_metrics["recall"] >= 0.85
assert dns_queries["entropy"].between(0, 6).all()
assert ranked_dns["risk_probability"].is_monotonic_decreasing
assert ranked_dns.head(10)["tunnel"].mean() >= 0.8
print("Checks passed: high recall, bounded entropy, sorted ranking, and a precise top alert set.")


Checks passed: high recall, bounded entropy, sorted ranking, and a precise top alert set.


## Next Steps

        - Aggregate features by client, registered domain, and time window.
- Whitelist known high-entropy services such as CDNs and security products.
- Monitor model drift as domain-generation behavior changes.
